# 03 · Agentic Workflows

**Goal:** see how an *agent* differs from a single prompt. An agent carries
**state (memory)**, applies **validation/routing rules** to decide what to do
next, and only then produces its final output. This mirrors the CareerForge
state machine (`state.py` + `main.py`).

## 1. State: the agent's memory

We model an interview session as a dataclass. Every turn mutates this state — it
is the single source of truth.

In [1]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class InterviewState:
    questions: List[str]
    answers: List[str] = field(default_factory=list)
    scores: List[int] = field(default_factory=list)
    index: int = 0

    def current(self) -> Optional[str]:
        return self.questions[self.index] if self.index < len(self.questions) else None

    def done(self) -> bool:
        return self.index >= len(self.questions)

state = InterviewState(questions=[
    "What is gradient descent?",
    "Explain overfitting.",
    "Why use a validation set?",
])
print("First question:", state.current())

First question: What is gradient descent?


## 2. Validation rules modify state before acting

The agent doesn't blindly advance. It **validates** input (e.g. reject empty
answers), grades it, records the result, then moves the pointer. Routing decisions
are driven by the mutated state.

In [2]:
import os, json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def grade(question: str, answer: str) -> int:
    r = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content":
             'Grade the answer 0-100 for accuracy. Respond JSON: {"score": int}.'},
            {"role": "user", "content": f"Q: {question}\nA: {answer}"},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return int(json.loads(r.choices[0].message.content)["score"])

class ValidationError(Exception):
    pass

def submit(state: InterviewState, answer: str) -> str:
    if state.done():
        raise ValidationError("Interview already finished.")
    if not answer.strip():
        raise ValidationError("Empty answer rejected — please respond.")

    q = state.current()
    score = grade(q, answer)          # call the tool (LLM grader)
    state.answers.append(answer)      # mutate memory
    state.scores.append(score)
    state.index += 1                  # advance the pointer (routing)
    return f"Scored {score}/100. " + ("Next question ready." if not state.done() else "Interview complete.")

## 3. Drive the state machine turn by turn

In [3]:
demo_answers = [
    "Gradient descent iteratively updates parameters against the loss gradient.",
    "Overfitting is when a model memorizes training data and fails to generalize.",
    "A validation set gives an unbiased estimate to tune hyperparameters.",
]

for a in demo_answers:
    print(">>", state.current())
    print("  ", submit(state, a))

print("\nFinal scores:", state.scores)
print("Average:", sum(state.scores) / len(state.scores))

>> What is gradient descent?
   Scored 95/100. Next question ready.
>> Explain overfitting.
   Scored 80/100. Next question ready.
>> Why use a validation set?
   Scored 80/100. Interview complete.

Final scores: [95, 80, 80]
Average: 85.0


## 4. Show the validation guard actually guards

Empty answers and post-completion submissions are rejected — the agent's rules,
not the LLM, enforce this.

In [4]:
fresh = InterviewState(questions=["Define recall."])
for bad in ["", "   "]:
    try:
        submit(fresh, bad)
    except ValidationError as e:
        print("Rejected:", e)

Rejected: Empty answer rejected — please respond.
Rejected: Empty answer rejected — please respond.


## 5. Routing: pick the next agent from state

A supervisor inspects state and dispatches to the right specialist. This is
exactly how `main.py` routes `/submit-answer` between the verbal and writing phases.

In [5]:
def route(state: InterviewState) -> str:
    if not state.done():
        return "VERBAL_AGENT  -> ask next question"
    avg = sum(state.scores) / len(state.scores) if state.scores else 0
    if avg >= 70:
        return "PATHWAY_AGENT -> candidate strong, build advanced plan"
    return "PATHWAY_AGENT -> candidate has gaps, build remedial plan"

print(route(state))

PATHWAY_AGENT -> candidate strong, build advanced plan


### Recap
- **State** is explicit, persistent memory the agent mutates each turn.
- **Validation rules** run *before* acting and can veto a transition.
- **Tools** (here, the LLM grader) are called to enrich state.
- **Routing** reads the mutated state to choose the next step/agent.

You now have every concept behind `02_career_forge_agent/`. Open `main.py` to see
the same pattern as a FastAPI service.